# Generate Precomputed Magnetic Field (GPU Analytical)

This notebook generates the precomputed magnetic field lookup table using the **GPU-accelerated analytical** formulas (Engel-Herbert & Hesjedal).

## Advantages over Magpylib:
- **5-10x faster** with GPU acceleration
- **No external dependencies** (uses CuPy)
- **Identical results** (matches magpylib within μT precision)

## Example Parameters (modify as needed):
- **Material**: N52 NdFeB
- **Remanence (Br)**: 1.48 T
- **Demagnetization Factor**: 0.9836
- **Magnet Size**: 6.35 mm
- **Grid Range**: ±225 mm
- **Step Size**: 1 mm

**All values below are examples - adjust to your specific magnet and grid requirements.**

## Important Note:
The demagnetization correction is **applied once** during precomputation.
When loading this field, **DO NOT** apply demag_factor again!

In [1]:
import numpy as np
import cupy as cp
import time
from datetime import datetime
import os
import sys

# Import analytical GPU module
sys.path.insert(0, 'Magnet_design')
from analytical_magnet_gpu import cuboid_field_gpu

print("="*80)
print("GPU ANALYTICAL PRECOMPUTED FIELD GENERATOR")
print("="*80)
print(f"CuPy version: {cp.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")

GPU ANALYTICAL PRECOMPUTED FIELD GENERATOR
CuPy version: 13.4.1
NumPy version: 1.26.4
Date: 2026-10-05 20:14:02
GPU: NVIDIA GeForce RTX 4070


## Configure Parameters (Example Values - Modify as Needed)

In [2]:
# ============================================================
# EXAMPLE VALUES - Modify these for your specific application
# ============================================================

# Material properties (example: N52 NdFeB)
Br = 1.48  # T - Remanence (adjust for your magnet grade)
demag_factor = 0.9836  # Demagnetization correction (geometry dependent)
effective_magnetization = Br * demag_factor  # T

# Magnet geometry (example: 6.35mm cube)
magnet_size_mm = 6.35  # mm - Change to your magnet size
magnet_size_m = magnet_size_mm * 1e-3  # m

# Grid parameters (example: 450mm span at 1mm resolution)
grid_range_mm = 100  # mm (±100 mm = 200mm   total) - Adjust for your FOV
step_size_mm = 1.0   # mm - Smaller = more accurate but larger file

# Batch size for GPU processing
batch_size = 500000  # points per batch - reduce if GPU runs out of memory
# Recommended batch sizes:
#   1,000,000 = fastest, needs ~8 GB VRAM
#   500,000 = balanced, needs ~4 GB VRAM (DEFAULT)
#   100,000 = safe for smaller GPUs

# ============================================================
# Convert to meters (don't modify)
# ============================================================
grid_range_m = grid_range_mm * 1e-3
step_size_m = step_size_mm * 1e-3

print("\n" + "="*80)
print("CONFIGURATION (Example Values - Modify Above)")
print("="*80)
print(f"\nMaterial Properties:")
print(f"  • Remanence (Br): {Br} T")
print(f"  • Demagnetization Factor: {demag_factor}")
print(f"  • Effective Magnetization: {effective_magnetization:.6f} T")
print(f"\nMagnet Geometry:")
print(f"  • Size: {magnet_size_mm} mm cube")
print(f"  • Polarization: +Y axis (0, {effective_magnetization:.6f}, 0)")
print(f"\nGrid Parameters:")
print(f"  • Range: ±{grid_range_mm} mm ({grid_range_mm*2} mm total)")
print(f"  • Step Size: {step_size_mm} mm")

# Calculate grid size
n_points_per_axis = int(2 * grid_range_m / step_size_m)
total_points = n_points_per_axis ** 3
n_batches = (total_points + batch_size - 1) // batch_size

print(f"  • Points per axis: {n_points_per_axis}")
print(f"  • Total grid points: {total_points:,}")
print(f"\nMemory Estimates:")
print(f"  • Final file size (compressed): ~{total_points * 3 * 8 / (1024**3) * 0.6:.2f} GB")
print(f"  • Number of batches: {n_batches:,}")
print(f"  • Batch size: {batch_size:,} points")
print("="*80)


CONFIGURATION (Example Values - Modify Above)

Material Properties:
  • Remanence (Br): 1.48 T
  • Demagnetization Factor: 0.9836
  • Effective Magnetization: 1.455728 T

Magnet Geometry:
  • Size: 6.35 mm cube
  • Polarization: +Y axis (0, 1.455728, 0)

Grid Parameters:
  • Range: ±100 mm (200 mm total)
  • Step Size: 1.0 mm
  • Points per axis: 200
  • Total grid points: 8,000,000

Memory Estimates:
  • Final file size (compressed): ~0.11 GB
  • Number of batches: 16
  • Batch size: 500,000 points


## Generate Grid

In [3]:
print("\nGenerating grid...")
start_time = time.time()

# Create 1D grid
grid_range = np.arange(-grid_range_m, grid_range_m, step_size_m)
print(f"  • 1D grid: {len(grid_range)} points")

# Create 3D meshgrid
X, Y, Z = np.meshgrid(grid_range, grid_range, grid_range, indexing='ij')
Ref_grid = X  # Save for output

# Flatten to points array
points = np.stack((X, Y, Z), axis=-1).reshape(-1, 3)

grid_time = time.time() - start_time
print(f"✓ Grid generated in {grid_time:.2f} seconds")
print(f"  • Shape: {X.shape}")
print(f"  • Total points: {len(points):,}")
print(f"  • Points array size: {points.nbytes / (1024**2):.1f} MB")

# Clean up large arrays we don't need anymore
del Y, Z
print(f"  • Cleaned up temporary arrays")


Generating grid...
  • 1D grid: 200 points
✓ Grid generated in 0.07 seconds
  • Shape: (200, 200, 200)
  • Total points: 8,000,000
  • Points array size: 183.1 MB
  • Cleaned up temporary arrays


## Create Magnet (Using Parameters Above)

In [4]:
print("\nConfiguring magnet parameters...")
print(f"  • Polarization: (0, {effective_magnetization:.6f}, 0) T")
print(f"  • Dimensions: ({magnet_size_mm}, {magnet_size_mm}, {magnet_size_mm}) mm")
print(f"  • Position: Origin (0, 0, 0)")

# Store parameters for analytical calculation
polarization = [0, effective_magnetization, 0]  # Y-axis
dimensions = [magnet_size_m, magnet_size_m, magnet_size_m]

print("✓ Parameters configured for GPU analytical calculation")


Configuring magnet parameters...
  • Polarization: (0, 1.455728, 0) T
  • Dimensions: (6.35, 6.35, 6.35) mm
  • Position: Origin (0, 0, 0)
✓ Parameters configured for GPU analytical calculation


## Calculate Field in Batches

**This is the long step** - progress updates every 10 batches.

In [5]:
print("\n" + "="*80)
print("CALCULATING MAGNETIC FIELD (GPU ANALYTICAL)")
print("="*80)
print(f"Processing {total_points:,} points in {n_batches:,} batches...\n")

# Pre-allocate output array
Bxyz = np.zeros((total_points, 3), dtype=np.float64)

calc_start = time.time()

for i in range(0, total_points, batch_size):
    end = min(i + batch_size, total_points)
    batch_num = i // batch_size + 1
    
    # Calculate field for this batch using GPU analytical
    batch_points = cp.asarray(points[i:end], dtype=cp.float64)
    B_gpu = cuboid_field_gpu(batch_points, polarization, dimensions)
    Bxyz[i:end] = cp.asnumpy(B_gpu)
    
    # Free GPU memory
    del batch_points, B_gpu
    cp.get_default_memory_pool().free_all_blocks()
    
    # Progress update every 10 batches or at end
    current_time = time.time()
    if batch_num % 10 == 0 or end == total_points:
        elapsed = current_time - calc_start
        progress_pct = end / total_points * 100
        rate = end / elapsed if elapsed > 0 else 0
        eta = (total_points - end) / rate if rate > 0 else 0
        
        print(f"  Batch {batch_num:>4}/{n_batches}: "
              f"{end:>10,}/{total_points:,} points "
              f"({progress_pct:>5.1f}%) | "
              f"Rate: {rate:>8,.0f} pts/s | "
              f"ETA: {eta/60:>5.1f} min")

calc_time = time.time() - calc_start

print(f"\n✓ Field calculation complete!")
print(f"  • Total time: {calc_time:.1f} seconds ({calc_time/60:.1f} minutes)")
print(f"  • Average rate: {total_points/calc_time:,.0f} points/second")
print("="*80)


CALCULATING MAGNETIC FIELD (GPU ANALYTICAL)
Processing 8,000,000 points in 16 batches...

  Batch   10/16:  5,000,000/8,000,000 points ( 62.5%) | Rate: 10,269,580 pts/s | ETA:   0.0 min
  Batch   16/16:  8,000,000/8,000,000 points (100.0%) | Rate: 11,422,175 pts/s | ETA:   0.0 min

✓ Field calculation complete!
  • Total time: 0.7 seconds (0.0 minutes)
  • Average rate: 11,420,181 points/second


## Verify Field

In [6]:
print("\n" + "="*80)
print("FIELD VERIFICATION")
print("="*80)

# Calculate field magnitude
field_magnitude = np.linalg.norm(Bxyz, axis=1)

print(f"\nArray Properties:")
print(f"  • Shape: {Bxyz.shape}")
print(f"  • Data type: {Bxyz.dtype}")
print(f"  • Memory: {Bxyz.nbytes / (1024**3):.2f} GB")

print(f"\nField Magnitude Statistics:")
print(f"  • Mean: {np.mean(field_magnitude):.6f} T = {np.mean(field_magnitude)*1000:.4f} mT")
print(f"  • Max:  {np.max(field_magnitude):.6f} T = {np.max(field_magnitude)*1000:.2f} mT")
print(f"  • Min:  {np.min(field_magnitude):.6f} T = {np.min(field_magnitude)*1000:.6f} mT")
print(f"  • Std:  {np.std(field_magnitude):.6f} T = {np.std(field_magnitude)*1000:.4f} mT")

# Check field at origin
center_idx = len(points) // 2
print(f"\nField Near Origin (index {center_idx}):")
print(f"  • Position: ({points[center_idx,0]*1000:.2f}, {points[center_idx,1]*1000:.2f}, {points[center_idx,2]*1000:.2f}) mm")
print(f"  • Field (x,y,z): ({Bxyz[center_idx,0]*1000:.4f}, {Bxyz[center_idx,1]*1000:.4f}, {Bxyz[center_idx,2]*1000:.4f}) mT")
print(f"  • Magnitude: {field_magnitude[center_idx]*1000:.4f} mT")

print(f"\nSanity Checks:")
print(f"  • Input magnetization: {effective_magnetization:.6f} T")
print(f"  • Max field / magnetization: {np.max(field_magnitude)/effective_magnetization:.4f}")
print(f"    (Should be < 1.0, typically ~0.8-0.9 for near-surface points)")

# Check if values are reasonable
if np.max(field_magnitude) > 2.0:
    print("  ⚠️  WARNING: Max field > 2 T seems too high!")
elif np.max(field_magnitude) < 0.1:
    print("  ⚠️  WARNING: Max field < 0.1 T seems too low!")
else:
    print("  ✓ Field values look reasonable")

print("="*80)


FIELD VERIFICATION

Array Properties:
  • Shape: (8000000, 3)
  • Data type: float64
  • Memory: 0.18 GB

Field Magnitude Statistics:
  • Mean: 0.000255 T = 0.2553 mT
  • Max:  1.220973 T = 1220.97 mT
  • Min:  0.000008 T = 0.008073 mT
  • Std:  0.007370 T = 7.3696 mT

Field Near Origin (index 4000000):
  • Position: (0.00, -100.00, -100.00) mm
  • Field (x,y,z): (0.0000, 0.0052, 0.0157) mT
  • Magnitude: 0.0166 mT

Sanity Checks:
  • Input magnetization: 1.455728 T
  • Max field / magnetization: 0.8387
    (Should be < 1.0, typically ~0.8-0.9 for near-surface points)
  ✓ Field values look reasonable


## Save Precomputed Field

In [7]:
# Generate descriptive filename with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"Precalc_field_{int(grid_range_mm*2)}mm_grid_{int(step_size_mm)}mm_step_Br{Br}_demag{demag_factor}_{timestamp}.npz"

print(f"\nSaving precomputed field...")
print(f"  • Filename: {filename}")
print(f"  • Compression: enabled (npz_compressed)")

save_start = time.time()

# Save ONLY the essential arrays (matching existing file format)
np.savez_compressed(
    filename,
    Bxyzvalue=Bxyz,      # (N, 3) field vectors
    Ref_grid=Ref_grid,   # (450, 450, 450) X-coordinates
    Grid_dim=grid_range, # (450,) 1D grid coordinates
    step_size=step_size_m # scalar step size
)

save_time = time.time() - save_start
file_size_gb = os.path.getsize(filename) / (1024**3)

print(f"✓ File saved in {save_time:.1f} seconds")
print(f"  • Size: {file_size_gb:.2f} GB")
print(f"  • Compression ratio: {Bxyz.nbytes / os.path.getsize(filename):.2f}x")
print(f"\n✅ Output format matches existing file (4 arrays only)")
print(f"  • Bxyzvalue: {Bxyz.shape}, {Bxyz.dtype}")
print(f"  • Ref_grid: {Ref_grid.shape}, {Ref_grid.dtype}")
print(f"  • Grid_dim: {grid_range.shape}, {grid_range.dtype}")
print(f"  • step_size: scalar, {type(step_size_m)}")


Saving precomputed field...
  • Filename: Precalc_field_200mm_grid_1mm_step_Br1.48_demag0.9836_20261005_201403.npz
  • Compression: enabled (npz_compressed)
✓ File saved in 3.3 seconds
  • Size: 0.13 GB
  • Compression ratio: 1.34x

✅ Output format matches existing file (4 arrays only)
  • Bxyzvalue: (8000000, 3), float64
  • Ref_grid: (200, 200, 200), float64
  • Grid_dim: (200,), float64
  • step_size: scalar, <class 'float'>


## Create README Documentation

In [8]:
doc_filename = filename.replace('.npz', '_README.txt')

doc_content = f"""PRECOMPUTED MAGNETIC FIELD
==========================

File: {filename}
Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

PARAMETERS
----------
Remanence (Br):          {Br} T
Demag Factor:            {demag_factor}
Effective Magnetization: {effective_magnetization:.6f} T
Magnet Size:             {magnet_size_mm} mm cube
Grid Range:              ±{grid_range_mm} mm
Step Size:               {step_size_mm} mm
Total Points:            {len(points):,}

ARRAYS IN FILE
--------------
Bxyzvalue   Magnetic field vectors (N×3) [Tesla]
Ref_grid    Reference X-coordinate grid
Grid_dim    1D grid coordinates [meters]
step_size   Grid spacing [meters]

USAGE
-----
B = np.load('{filename}')
field = B['Bxyzvalue']  # Already includes demag correction

⚠️  DO NOT multiply by demag_factor again - it's already applied!
"""

with open(doc_filename, 'w') as f:
    f.write(doc_content)

print(f"✓ README saved: {doc_filename}")

✓ README saved: Precalc_field_200mm_grid_1mm_step_Br1.48_demag0.9836_20261005_201403_README.txt


## Summary

In [9]:
print("\n" + "="*80)
print("GENERATION COMPLETE ✅")
print("="*80)
print(f"\nFiles Created:")
print(f"  1. {filename}")
print(f"     Size: {file_size_gb:.2f} GB")
print(f"     Contains: Precomputed magnetic field with metadata")
print(f"\n  2. {doc_filename}")
print(f"     Size: {os.path.getsize(doc_filename)/1024:.1f} KB")
print(f"     Contains: Complete documentation and usage instructions")

print(f"\nKey Parameters:")
print(f"  • Effective Magnetization: {effective_magnetization:.6f} T")
print(f"  • Grid: {n_points_per_axis}³ = {total_points:,} points")
print(f"  • Step Size: {step_size_mm} mm")
print(f"  • Computation Time: {calc_time/60:.1f} minutes")

print(f"\n⚠️  IMPORTANT REMINDER:")
print(f"   When loading this field, DO NOT multiply by demag_factor!")
print(f"   The correction is already applied.")

print(f"\nUsage:")
print(f"   B = np.load('{filename}')")
print(f"   field = B['Bxyzvalue'].astype(np.float32)  # ✅ Correct")
print("="*80)

# Clean up large arrays
del Bxyz, points, Ref_grid, field_magnitude
print("\n✓ Cleaned up memory")


GENERATION COMPLETE ✅

Files Created:
  1. Precalc_field_200mm_grid_1mm_step_Br1.48_demag0.9836_20261005_201403.npz
     Size: 0.13 GB
     Contains: Precomputed magnetic field with metadata

  2. Precalc_field_200mm_grid_1mm_step_Br1.48_demag0.9836_20261005_201403_README.txt
     Size: 0.8 KB
     Contains: Complete documentation and usage instructions

Key Parameters:
  • Effective Magnetization: 1.455728 T
  • Grid: 200³ = 8,000,000 points
  • Step Size: 1.0 mm
  • Computation Time: 0.0 minutes

⚠️  IMPORTANT REMINDER:
   When loading this field, DO NOT multiply by demag_factor!
   The correction is already applied.

Usage:
   B = np.load('Precalc_field_200mm_grid_1mm_step_Br1.48_demag0.9836_20261005_201403.npz')
   field = B['Bxyzvalue'].astype(np.float32)  # ✅ Correct

✓ Cleaned up memory


## Verify Output Format Matches Existing File